In [9]:
import torch
import pandas as pd
from torch.utils.data import DataLoader
from optional_train import DigitNet, DfToDataset

In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используется {device}")

config = {
    'channels': [32, 64],
    'kernel_size': 3,
    'dropout': 0.3,
    'mlp_dim': 128,
    'pool': 2,
    'stride': 2
}

Используется cuda


In [11]:
print("Загрузка моделей...")
all_models = []

for fold in range(5):
    model = DigitNet(config).to(device)
    model.load_state_dict(torch.load(f'models/model_fold_{fold + 1}.pt', map_location=device))
    model.eval()
    all_models.append(model)

print("Модели загружены")

Загрузка моделей...
Модели загружены


In [12]:
X_df = pd.read_csv("data/test.csv")

In [13]:
X_test = DfToDataset(X_df, is_test=True)
test_loader = DataLoader(X_test, batch_size=64, shuffle=False)

final_predictions = []

with torch.no_grad():
    for image in test_loader:
        image = image.to(device)

        batch_logits = []

        for model in all_models:
            logits = model(image)
            batch_logits.append(logits)

        mean_logits = torch.stack(batch_logits).mean(dim=0)
        preds = torch.argmax(mean_logits, dim=1)
        final_predictions.extend(preds.cpu().numpy())

submission = pd.DataFrame({
    'ImageId': range(1, len(final_predictions) + 1),
    'Label': final_predictions
})

submission.to_csv('results/submission.csv', index=False)
print("Файл submission.csv успешно сохранен!")

Файл submission.csv успешно сохранен!
